In [2]:
import pandas as pd
import requests
import time

from bs4 import BeautifulSoup
from urllib.parse import urljoin

In [3]:
url = "https://en.wikipedia.org/wiki/Lists_of_American_colleges_and_universities"

headers = {"User-Agent": "WebDataScience/1.0 (ansc4435@colorado.edu)"}

response = requests.get(url, headers = headers)

print(response.status_code)

200


In [4]:
soup = BeautifulSoup(response.text, "lxml")

In [5]:
states = [
    "Alabama", "Alaska", "Arizona", "Arkansas", "California",
    "Colorado", "Connecticut", "Delaware", "Florida", "Georgia",
    "Hawaii", "Idaho", "Illinois", "Indiana", "Iowa", "Kansas",
    "Kentucky", "Louisiana", "Maine", "Maryland", "Massachusetts",
    "Michigan", "Minnesota", "Mississippi", "Missouri", "Montana",
    "Nebraska", "Nevada", "New Hampshire", "New Jersey",
    "New Mexico", "New York", "North Carolina", "North Dakota",
    "Ohio", "Oklahoma", "Oregon", "Pennsylvania", "Rhode Island",
    "South Carolina", "South Dakota", "Tennessee", "Texas", "Utah",
    "Vermont", "Virginia", "Washington", "West Virginia",
    "Wisconsin", "Wyoming"
]

state_links = []
for link in soup.select("#mw-content-text a[href]"):
    text = link.get_text(strip=True)
    href = link["href"]
    if text in states and "List_of_colleges_and_universities_in_" in href:
        state_links.append({"state": text, "url": urljoin(url, href)})

states_df = (pd.DataFrame(state_links)
               .drop_duplicates(subset="state")
               .reset_index(drop=True))

print(states_df)

             state                                                url
0      Connecticut  https://en.wikipedia.org/wiki/List_of_colleges...
1            Maine  https://en.wikipedia.org/wiki/List_of_colleges...
2    Massachusetts  https://en.wikipedia.org/wiki/List_of_colleges...
3    New Hampshire  https://en.wikipedia.org/wiki/List_of_colleges...
4       New Jersey  https://en.wikipedia.org/wiki/List_of_colleges...
5         New York  https://en.wikipedia.org/wiki/List_of_colleges...
6     Pennsylvania  https://en.wikipedia.org/wiki/List_of_colleges...
7     Rhode Island  https://en.wikipedia.org/wiki/List_of_colleges...
8          Vermont  https://en.wikipedia.org/wiki/List_of_colleges...
9         Illinois  https://en.wikipedia.org/wiki/List_of_colleges...
10         Indiana  https://en.wikipedia.org/wiki/List_of_colleges...
11            Iowa  https://en.wikipedia.org/wiki/List_of_colleges...
12          Kansas  https://en.wikipedia.org/wiki/List_of_colleges...
13        Michigan  

In [6]:
def extract_schools(soup, state, debug=False):

    schools = []

    content = soup.select_one(
        "#mw-content-text .mw-parser-output"
    )

    if content is None:
        print(f"Could not find article content for {state}")
        return schools

    excluded_sections = [
        "see also",
        "references",
        "external links",
        "footnotes",
        "notes"
    ]

    excluded_url_parts = [
        "/wiki/File:",
        "/wiki/Category:",
        "/wiki/Special:",
        "/wiki/Help:",
        "/wiki/Portal:"
    ]

    def get_section(element):
        heading = element.find_previous(
            ["h2", "h3", "h4"]
        )

        if heading:
            return heading.get_text(" ", strip=True)

        return "Uncategorized"

    def get_valid_wikipedia_link(cell):
        if cell is None:
            return None

        for link in cell.find_all("a", href=True):
            href = link["href"]

            if href.startswith("/wiki/"):
                full_url = urljoin(
                    "https://en.wikipedia.org",
                    href
                )

            elif href.startswith(
                "https://en.wikipedia.org/wiki/"
            ):
                full_url = href

            else:
                continue

            if any(
                excluded in full_url
                for excluded in excluded_url_parts
            ):
                continue

            if "action=edit" in full_url:
                continue

            return link

        return None

    def save_school(name, link, section):
        name = name.strip()

        if not name:
            return

        wikipedia_url = None

        if link is not None:
            href = link["href"]

            if href.startswith("/wiki/"):
                wikipedia_url = urljoin(
                    "https://en.wikipedia.org",
                    href
                )

            elif href.startswith(
                "https://en.wikipedia.org/wiki/"
            ):
                wikipedia_url = href

        schools.append({
            "university": name,
            "state": state,
            "wikipedia_url": wikipedia_url
        })

    for table in content.select("table.wikitable"):

        section = get_section(table)

        if section.lower() in excluded_sections:
            continue

        rows = table.find_all("tr")

        name_column = None
        header_index = None

        for row_index, row in enumerate(rows):

            cells = row.find_all(
                ["th", "td"],
                recursive=False
            )

            headers = [
                cell.get_text(" ", strip=True).lower()
                for cell in cells
            ]

            for index, header in enumerate(headers):

                if header in [
                    "name",
                    "school",
                    "institution"
                ]:
                    name_column = index
                    header_index = row_index
                    break

            if name_column is not None:
                break

        if name_column is None:
            continue

        for row in rows[header_index + 1:]:

            cells = row.find_all(
                ["th", "td"],
                recursive=False
            )

            if len(cells) <= name_column:
                continue

            name_cell = cells[name_column]

            name = name_cell.get_text(
                " ",
                strip=True
            )

            link = get_valid_wikipedia_link(name_cell)

            save_school(name, link, section)


    for listing in content.find_all(["ul", "ol"]):

        if listing.find_parent("table"):
            continue

        section = get_section(listing)

        if section.lower() in excluded_sections:
            continue

        for item in listing.find_all(
            "li",
            recursive=False
        ):

            links = [
                link for link in item.find_all(
                    "a",
                    href=True
                )
                if link.find_parent("li") == item
            ]

            valid_link = None

            for link in links:

                href = link.get("href", "")

                if (
                    href.startswith("/wiki/")
                    or href.startswith(
                        "https://en.wikipedia.org/wiki/"
                    )
                ) and not any(
                    excluded in href
                    for excluded in excluded_url_parts
                ) and "action=edit" not in href:

                    valid_link = link
                    break

            if valid_link is not None:

                name = valid_link.get_text(
                    " ",
                    strip=True
                )

                save_school(name, valid_link, section)

    unique_schools = []
    seen = set()

    for school in schools:

        if school["wikipedia_url"]:

            key = school["wikipedia_url"].split("#")[0]

        else:

            key = (
                school["state"],
                school["university"].lower()
            )

        if key not in seen:

            seen.add(key)
            unique_schools.append(school)

    if debug:
        print(
            f"{state}: Found {len(unique_schools)} "
            "unique institutions"
        )

    return unique_schools

In [7]:
all_schools = []
scrape_errors = []

for _, row in states_df.iterrows():

    state = row["state"]
    state_url = row["url"]

    print(f"Scraping {state}...")

    try:
        response = requests.get(
            state_url,
            headers=headers,
            timeout=30
        )

        response.raise_for_status()

        state_soup = BeautifulSoup(
            response.text,
            "lxml"
        )

        schools = extract_schools(
            state_soup,
            state,
            debug=False
        )

        all_schools.extend(schools)

        print(f"Found {len(schools)} institutions in {state}")

    except Exception as e:
        print(f"Error scraping {state}: {e}")

        scrape_errors.append({
            "state": state,
            "error": str(e)
        })

    time.sleep(1)

national_df = pd.DataFrame(all_schools)

print("\nScraping finished.")
print("Total records:", len(national_df))
print("States successfully scraped:", national_df["state"].nunique())
print("States with errors:", len(scrape_errors))

Scraping Connecticut...
Found 50 institutions in Connecticut
Scraping Maine...
Found 48 institutions in Maine
Scraping Massachusetts...
Found 200 institutions in Massachusetts
Scraping New Hampshire...
Found 25 institutions in New Hampshire
Scraping New Jersey...
Found 119 institutions in New Jersey
Scraping New York...
Found 280 institutions in New York
Scraping Pennsylvania...
Found 232 institutions in Pennsylvania
Scraping Rhode Island...
Found 21 institutions in Rhode Island
Scraping Vermont...
Found 35 institutions in Vermont
Scraping Illinois...
Found 142 institutions in Illinois
Scraping Indiana...
Found 80 institutions in Indiana
Scraping Iowa...
Found 73 institutions in Iowa
Scraping Kansas...
Found 59 institutions in Kansas
Scraping Michigan...
Found 104 institutions in Michigan
Scraping Minnesota...
Found 99 institutions in Minnesota
Scraping Missouri...
Found 88 institutions in Missouri
Scraping Nebraska...
Found 50 institutions in Nebraska
Scraping North Dakota...
Found 24

In [8]:
national_df_backup = national_df.copy()

print("Backup created.")

Backup created.


In [9]:
file_mask = national_df["wikipedia_url"].str.contains(
    "/wiki/File:",
    na=False
)

national_df.loc[file_mask, "wikipedia_url"] = None

invalid_names = [
    "ISBN",
    "Staten Island",
    "4-year institution USG average",
    "National average",
    "Total"
]

national_df = national_df[
    ~national_df["university"].isin(invalid_names)
].copy()


print("Cleanup complete.")
print("Remaining records:", len(national_df))
print(
    "Missing URLs:",
    national_df["wikipedia_url"].isna().sum()
)

Cleanup complete.
Remaining records: 4216
Missing URLs: 360


In [10]:

print("Total institutions:", len(national_df))

print(
    "Unique institution names:",
    national_df["university"].nunique()
)

print(
    "Unique Wikipedia URLs:",
    national_df["wikipedia_url"].nunique()
)

print(
    "Missing URLs:",
    national_df["wikipedia_url"].isna().sum()
)

print("\nInstitutions per state:")
print(
    national_df.groupby("state")["university"]
    .count()
    .sort_values(ascending=False)
)

print("\nRemaining Wikipedia File URLs:")
print(
    national_df[
        national_df["wikipedia_url"].str.contains(
            "/wiki/File:",
            na=False
        )
    ][["university", "state", "wikipedia_url"]]
)

Total institutions: 4216
Unique institution names: 4026
Unique Wikipedia URLs: 3684
Missing URLs: 360

Institutions per state:
state
California        320
New York          278
Pennsylvania      232
Massachusetts     200
Texas             181
Florida           167
Illinois          142
Virginia          140
Georgia           134
North Carolina    130
Ohio              121
New Jersey        119
Washington        111
Michigan          104
Oregon            103
Minnesota          98
Tennessee          93
Missouri           88
Wisconsin          87
Maryland           81
Indiana            80
Alabama            79
Arizona            74
Iowa               73
Kentucky           72
South Carolina     66
Kansas             59
Louisiana          58
Oklahoma           55
Arkansas           53
West Virginia      51
Nebraska           50
Connecticut        49
Maine              48
New Mexico         46
Colorado           44
Utah               37
Vermont            35
Mississippi        33
South Dak

In [11]:
national_df.to_csv(
    "american_colleges_universities.csv",
    index=False
)

print("Dataset saved successfully.")

Dataset saved successfully.


In [12]:
missing_by_state = (
    national_df[national_df["wikipedia_url"].isna()]
    .groupby("state")["university"]
    .count()
    .sort_values(ascending=False)
)

print(missing_by_state.to_string())

state
California        73
New Jersey        47
Massachusetts     44
Pennsylvania      31
Oregon            25
Virginia          24
Arizona           13
Nebraska          10
Connecticut        8
Minnesota          8
Maryland           8
New Mexico         7
New Hampshire      6
Illinois           6
Maine              6
Ohio               5
Nevada             4
Rhode Island       4
Oklahoma           3
West Virginia      3
Utah               3
Wisconsin          3
North Carolina     3
Indiana            3
North Dakota       2
Tennessee          2
Iowa               2
Idaho              1
Delaware           1
South Dakota       1
Arkansas           1
Vermont            1
Colorado           1
Michigan           1


In [13]:
print("Total institutions:", len(national_df))

print(
    "Missing Wikipedia URLs:",
    national_df["wikipedia_url"].isna().sum()
)

print(
    "Unique Wikipedia URLs:",
    national_df["wikipedia_url"].nunique()
)

Total institutions: 4216
Missing Wikipedia URLs: 360
Unique Wikipedia URLs: 3684


In [14]:
missing_urls_df = national_df[
    national_df["wikipedia_url"].isna()
][["university", "state"]].sort_values(
    by=["state", "university"]
).reset_index(drop=True)

print("Institutions missing Wikipedia URLs:", len(missing_urls_df))
display(missing_urls_df.head(20))

Institutions missing Wikipedia URLs: 360


,university,state
0,Arizona College of Nursing,Arizona
1,Arizona School of Acupuncture and Oriental Med...,Arizona
2,Cummings Graduate Institute for Behavioral Hea...,Arizona
3,Indigenous Bible College,Arizona
4,International Baptist College and Seminary,Arizona
5,Justice University,Arizona
6,National Paralegal College (Crestpoint Univers...,Arizona
7,Pathways College,Arizona
8,Phoenix Institute of Herbal Medicine & Acupunc...,Arizona
9,Refrigeration School Inc,Arizona


In [15]:
universities_with_urls_df = national_df[
    national_df["wikipedia_url"].notna()
    & national_df["wikipedia_url"].str.startswith(
        "https://en.wikipedia.org/wiki/"
    )
].copy()

print(
    "Institutions with Wikipedia URLs:",
    len(universities_with_urls_df)
)

print(
    "Unique Wikipedia URLs:",
    universities_with_urls_df["wikipedia_url"].nunique()
)

display(universities_with_urls_df.head(20))

Institutions with Wikipedia URLs: 3856
Unique Wikipedia URLs: 3684


,university,state,wikipedia_url
0,Albertus Magnus College,Connecticut,https://en.wikipedia.org/wiki/Albertus_Magnus_...
1,Central Connecticut State University,Connecticut,https://en.wikipedia.org/wiki/Central_Connecti...
2,Charter Oak State College,Connecticut,https://en.wikipedia.org/wiki/Charter_Oak_Stat...
3,Connecticut College,Connecticut,https://en.wikipedia.org/wiki/Connecticut_College
4,Connecticut State Community College,Connecticut,https://en.wikipedia.org/wiki/Connecticut_Stat...
5,Eastern Connecticut State University,Connecticut,https://en.wikipedia.org/wiki/Eastern_Connecti...
6,Fairfield University,Connecticut,https://en.wikipedia.org/wiki/Fairfield_Univer...
7,Goodwin University,Connecticut,https://en.wikipedia.org/wiki/Goodwin_University
8,Hartford International University for Religion...,Connecticut,https://en.wikipedia.org/wiki/Hartford_Interna...
9,Holy Apostles College and Seminary,Connecticut,https://en.wikipedia.org/wiki/Holy_Apostles_Co...


In [17]:
universities_with_urls_df.to_csv(
    "universities_with_wikipedia_urls.csv",
    index=False
)

print("University URL dataset saved.")

University URL dataset saved.


In [18]:
import re

def get_wikipedia_mascot(wikipedia_url):

    try:
        response = requests.get(
            wikipedia_url,
            headers=headers,
            timeout=30
        )

        response.raise_for_status()

        soup = BeautifulSoup(response.text, "lxml")

        infoboxes = soup.select("table.infobox")

        for infobox in infoboxes:

            for row in infobox.find_all("tr"):

                label = row.find("th")
                value = row.find("td")

                if label is None or value is None:
                    continue

                label_text = label.get_text(
                    " ",
                    strip=True
                ).lower()

                if "mascot" in label_text:
                    for reference in value.select("sup.reference"):
                        reference.decompose()

                    mascot = value.get_text(
                        " ",
                        strip=True
                    )

                    mascot = re.sub(
                        r"\\[\\d+\\]",
                        "",
                        mascot
                    )

                    mascot = re.sub(
                        r"\\s+",
                        " ",
                        mascot
                    ).strip()

                    return mascot if mascot else None

        return None

    except Exception as e:
        print(f"Error retrieving {wikipedia_url}: {e}")
        return None

In [20]:
test_df = (
    universities_with_urls_df
    .drop_duplicates(subset="wikipedia_url")
    .sample(n=10, random_state=42)
)

for _, row in test_df.iterrows():

    mascot = get_wikipedia_mascot(
        row["wikipedia_url"]
    )

    print("University:", row["university"])
    print("Mascot:", mascot)

    time.sleep(1)

University: Augusta University (AU)
Mascot: Augustus the Jaguar
University: Winthrop University
Mascot: Big Stuff
University: Pomona College
Mascot: Cecil the Sagehen
University: American Medical Missionary College
Mascot: None
University: Dropsie College for Hebrew and Cognate Learning
Mascot: None
University: Bowie State University
Mascot: Butch the Bulldog
University: University of Iowa
Mascot: Herky the Hawk
University: Whittier College
Mascot: Johnny Poet
University: The Bronx
Mascot: None
University: Wilmington University
Mascot: Wiley D. Wildcat


In [21]:
mascots_df = (
    universities_with_urls_df
    .drop_duplicates(subset="wikipedia_url")
    .copy()
)

mascots_df["mascot"] = None

for index, row in mascots_df.iterrows():

    mascot = get_wikipedia_mascot(
        row["wikipedia_url"]
    )

    mascots_df.at[index, "mascot"] = mascot

    print(
        f"{index + 1}/{len(mascots_df)}: "
        f"{row['university']} — {mascot or 'No mascot listed'}"
    )

    time.sleep(1)

print("\\nMascot collection complete!")

1/3684: Albertus Magnus College — No mascot field listed
2/3684: Central Connecticut State University — Kizer the Blue Devil
3/3684: Charter Oak State College — No mascot field listed
4/3684: Connecticut College — Camel ( Dromedary )
5/3684: Connecticut State Community College — No mascot field listed
6/3684: Eastern Connecticut State University — Willi The Warrior
7/3684: Fairfield University — Lucas the Stag
8/3684: Goodwin University — The Navigator
9/3684: Hartford International University for Religion and Peace — No mascot field listed
10/3684: Holy Apostles College and Seminary — No mascot field listed
11/3684: Mitchell College — No mascot field listed
12/3684: Post University — "Swoop" the Eagle
13/3684: Quinnipiac University — Boomer the Bobcat
14/3684: Sacred Heart University — "Big Red" the Pioneer
15/3684: Southern Connecticut State University — Otus the Owl
16/3684: Trinity College — Bantam
17/3684: United States Coast Guard Academy — Objee the Bear
18/3684: University of B

In [22]:
mascots_df.to_csv(
    "university_mascots.csv",
    index=False
)

print("CSV successfully saved!")

CSV successfully saved!


In [23]:
mascots_with_names_df = mascots_df[
    mascots_df["mascot"].notna()
    & (mascots_df["mascot"].str.strip() != "")
].copy()

In [24]:
print("Original rows:", len(mascots_df))
print("Rows with mascots:", len(mascots_with_names_df))
print("Rows removed:", len(mascots_df) - len(mascots_with_names_df))

Original rows: 3684
Rows with mascots: 1591
Rows removed: 2093


In [25]:
mascots_with_names_df.to_csv(
    "university_mascots_with_mascots_only.csv",
    index=False
)

print("Cleaned CSV saved!")

Cleaned CSV saved!
